In [1]:
from gensim.models      import Word2Vec
from model              import ConvAttnPool
import torch.nn.functional as F
import numpy as np
import torch
import os

In [2]:
word_model = Word2Vec.load('processed_full.w2v')
vocab_size, embed_size = word_model.wv.vectors.shape
print(f'{vocab_size=}', f'{embed_size=}')
embedding_table = torch.from_numpy(word_model.wv.vectors).type(torch.float32)
embedding_table = torch.concat([embedding_table,torch.zeros(size = (1,embed_size))], dim = 0)
print(f'{vocab_size=}', f'{embed_size=}') # Keep the since vocab_size refers to the last index, our padding index)

vocab_size=150853 embed_size=100
vocab_size=150853 embed_size=100


In [3]:
model = ConvAttnPool(
            drop_out       = 0.2,
            embed_table   = embedding_table,
            vocab_size    = vocab_size,
            num_of_filters = 15, # Filters in paper -> 10
            label_space    = 50, 
            kernel_size    = 5,
            embed_d        = embed_size)
assert(os.path.isfile('model.tar'))
saved_param = torch.load(f = 'model.tar')
print(model.load_state_dict(saved_param))
model.eval()
print(model)

<All keys matched successfully>
ConvAttnPool(
  (embed): Embedding(150854, 100, padding_idx=150853)
  (conv): Conv1d(100, 15, kernel_size=(5,), stride=(1,), padding=(2,))
  (U): Linear(in_features=15, out_features=50, bias=True)
  (final): Linear(in_features=15, out_features=50, bias=True)
  (embed_drop): Dropout(p=0.2, inplace=False)
)


In [4]:
X_train = np.load('X_train.npy')
X_test  = np.load('X_test.npy')

Y_train = np.load('Y_train.npy')
Y_test  = np.load('Y_test.npy')

X_train = torch.from_numpy(X_train).long()
Y_train = torch.from_numpy(Y_train).type(torch.float32)

X_test  = torch.from_numpy(X_test) .long()
Y_test  = torch.from_numpy(Y_test) .type(torch.float32)

Obtian a single sample and run it through our model. Obtains yhat and attn_map

In [ ]:
# for idx in range(n_samples):
#     instance = idx
#     X_sample, Y_sample = X_train[instance].unsqueeze(dim = 0), Y_train[instance]
#     yhat, attn = model(X_sample)
#     X_sample = X_sample.squeeze(dim = 0)
#     attn = attn.squeeze(dim = 0)
#     yhat = (F.sigmoid(yhat.squeeze(dim = 0)) > 0.5).long()
#     if (torch.equal(yhat,Y_sample)):
#         print(idx)
# """
# 43 68 89 92 98 112 123 126 129 150 163 174 233 255 261 262 287 288 296 317 320 329 335 350 351
# """

IndentationError: unexpected indent (2242452062.py, line 2)

In [5]:
Black   = "\033[0;90m"
IRed    = "\033[0;91m"
White   = "\033[0;37m"
IYellow = "\033[0;93m"
RColor  = "\033[0;0m"

In [10]:
instance = 35
X_sample, Y_sample = X_train[instance].unsqueeze(dim = 0), Y_train[instance]
yhat, attn = model(X_sample)
X_sample = X_sample.squeeze(dim = 0)
attn = attn.squeeze(dim = 0)
yhat = (F.sigmoid(yhat.squeeze(dim = 0)) > 0.5).long()

document_len = X_sample[X_sample != vocab_size].shape[0]
print(f'{document_len=}')
sample = X_sample[:document_len]
attn   = attn[:,:document_len]

print(yhat,Y_sample, sep='\n\n')

document_len=266
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0,
        0, 0])

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.])


--- 
Translate documnet

In [11]:
translated_doc = [word_model.wv.index_to_key[idx.item()] for idx in sample]

indices = (Y_sample == 1).nonzero().squeeze().tolist()
if type(indices) == int:
    indices = [indices]

word_per_row = 12
distance = 4

for label_idx in indices:
    # Grab the attention map for the labels
    attn_map  = F.softmax(attn[label_idx], dim = 0)
    max_token = torch.argmax(attn_map).item()
    print('_' * 100)
    print(f'{label_idx=}')

    for idx, token in enumerate(translated_doc):
        if idx == max_token:
            print(f'{IYellow}' + token, end = ' ')
            print(f'{RColor}', end = '')
        elif idx - distance < max_token < idx + distance:
            print(f'{White}' + token, end = ' ')
            print(f'{RColor}', end = '')
        else:
            print(token,end = ' ')

        if not (idx % word_per_row) and idx > 1:
            print()
    print()


____________________________________________________________________________________________________
label_idx=42
admission date discharge date date of birth sex f service neurosurgery history of 
present illness the patient is a year old female who was in 
her usual state of health until in the afternoon when she was 
shopping and all of the sudden shouted fell down to the floor 
yelling oh no the patient was transported to hospital hospital for care 
on ct scan at hospital hospital a large subarachnoid hemorrhage was seen 
the patient was then transferred to hospital1 for care past medical history 
none the rest of the history was unable to be obtained physical 
examination the patient was intubated the patient was unable to move extremities 
and the patient had no reflexes on examination studies ct scan of 
the head at hospital1 showed a massive subarachnoid hemorrhage completely filling the 
basal cistern in almost every visible sulcus and fissure a cta of 
the head was done which 